# Social Network Link Analysis (Student Activity) — Google Colab Notebook

This notebook helps you **build a graph from your own social network data** (LinkedIn / Instagram / any edge list) and perform **link analysis**.

✅ Works even if you don’t want to use personal data — you can generate a **synthetic network**.

---

## What you will do
1. Load (or create) a network dataset  
2. Build a graph (directed or undirected)  
3. Compute node-level and network-level measures  
4. Visualize the network and highlight key nodes  
5. Detect communities  
6. Mini “recommendation” task  
7. Robustness test (remove influential node)

---

## Privacy note
If you are using your real data, **do not submit names/emails**. Anonymize IDs before submission.


In [1]:
# Install / import libraries (Colab usually has these)
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from networkx.algorithms import community

# For reproducible layouts
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1) Choose your data source

Pick **ONE** of the following:

### A) LinkedIn Connections export
Download `Connections.csv` from LinkedIn and upload it to Colab (Files panel → Upload).

### B) Any edge list you create
Create a CSV called `edges.csv` with two columns:
- `source`, `target`
Each row is a connection/follow.

### C) No data? Generate a synthetic network
Use the built-in generator below.

> Tip: If your network is **large** (1000+ nodes), plotting the full graph may be slow. You can sample or plot a subgraph.


In [ ]:
from google.colab import files

# Upload your file(s) here if needed:
# - LinkedIn: Connections.csv
# - Edge list: edges.csv
uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))


## 2A) Load LinkedIn Connections.csv (optional)

LinkedIn exports typically include columns like **First Name, Last Name, Company, Position, Connected On**.
There isn't a direct "edge list" because it’s your ego-network: **you** connected to each person.

So we create an edge list:
- Node 0 = YOU
- Edge from YOU → Connection (or undirected YOU—Connection)

This still supports meaningful analysis (centrality, degree distribution, etc.), but it will look like a star unless you add second-hop connections (not available in standard export).

If you want a richer network structure, use Option **2B** with an edge list.


In [ ]:
# If you uploaded LinkedIn Connections.csv, set the filename below:
LINKEDIN_FILE = "Connections.csv"

import os
if os.path.exists(LINKEDIN_FILE):
    df = pd.read_csv(LINKEDIN_FILE)
    display(df.head())

    # Create anonymized node IDs
    # We'll map each connection to an integer ID (no names in outputs)
    df["person_id"] = range(1, len(df) + 1)  # start from 1
    YOU_ID = 0

    # Edge list: YOU connected to each person
    edges = pd.DataFrame({"source": YOU_ID, "target": df["person_id"].values})
    print("Edges shape:", edges.shape)
    display(edges.head())
else:
    print(f"'{LINKEDIN_FILE}' not found. Upload it first, or use Option 2B/2C.")


## 2B) Load your own edge list (recommended for richer graphs)

Create/upload `edges.csv` with:
- `source`, `target`

Example:
```
source,target
A,B
B,C
C,A
```

You can use numbers (1,2,3...) or anonymous labels.


In [ ]:
EDGE_FILE = "edges.csv"

import os
if os.path.exists(EDGE_FILE):
    edges = pd.read_csv(EDGE_FILE)
    assert {"source", "target"}.issubset(edges.columns), "edges.csv must have columns: source, target"
    display(edges.head())
    print("Edges shape:", edges.shape)
else:
    print(f"'{EDGE_FILE}' not found. Upload it, or use Option 2C.")


## 2C) Generate a synthetic social network (if you don't have data)

You can choose one:
- **Barablbási–Aert** (scale-free, hubs)
- **Watts–Strogatz** (small-world)
- **Erdős–Rényi** (random)

Set parameters and run.


In [ ]:
# Only run this if you don't have edges loaded above.
# It will create an 'edges' DataFrame.

USE_SYNTHETIC = True  # set True to generate
MODEL = "barabasi"    # "barabasi", "watts", "erdos"

n = 200               # number of nodes
if MODEL == "barabasi":
    m = 3             # new edges per node
elif MODEL == "watts":
    k = 6             # each node connected to k nearest neighbors
    p = 0.1           # rewiring probability
elif MODEL == "erdos":
    p = 0.03          # edge probability

import pandas as pd
import numpy as np
import networkx as nx

if USE_SYNTHETIC:
    if MODEL == "barabasi":
        Gtemp = nx.barabasi_albert_graph(n=n, m=m, seed=RANDOM_SEED)
    elif MODEL == "watts":
        Gtemp = nx.watts_strogatz_graph(n=n, k=k, p=p, seed=RANDOM_SEED)
    elif MODEL == "erdos":
        Gtemp = nx.erdos_renyi_graph(n=n, p=p, seed=RANDOM_SEED)
    else:
        raise ValueError("Unknown MODEL")

    edges = pd.DataFrame(list(Gtemp.edges()), columns=["source", "target"])
    print("Generated synthetic edges:", edges.shape)
    display(edges.head())
else:
    print("Synthetic generation is off.")


## 3) Build the graph

Choose:
- **Undirected** graph: mutual connection/friendship
- **Directed** graph: follows (A→B)

If you're unsure, use **undirected**.


In [ ]:
# Choose graph type
DIRECTED = False

# Basic validation
assert "edges" in globals(), "No 'edges' DataFrame found. Load/upload/generate edges first."
assert {"source", "target"}.issubset(edges.columns)

if DIRECTED:
    G = nx.from_pandas_edgelist(edges, "source", "target", create_using=nx.DiGraph())
else:
    G = nx.from_pandas_edgelist(edges, "source", "target", create_using=nx.Graph())

print(nx.info(G))
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())


## 4) Compute network-level measures

In [ ]:
def safe_diameter(graph):
    # Diameter only defined for connected graphs (or strongly connected for DiGraph)
    if graph.is_directed():
        if nx.is_strongly_connected(graph):
            return nx.diameter(graph)
        else:
            return None
    else:
        if nx.is_connected(graph):
            return nx.diameter(graph)
        else:
            return None

def average_shortest_path(graph):
    if graph.is_directed():
        if nx.is_strongly_connected(graph):
            return nx.average_shortest_path_length(graph)
        else:
            return None
    else:
        if nx.is_connected(graph):
            return nx.average_shortest_path_length(graph)
        else:
            return None

density = nx.density(G)
avg_degree = float(np.mean([d for _, d in G.degree()]))
diam = safe_diameter(G)
avg_path = average_shortest_path(G)

result = {
    "density": density,
    "average_degree": avg_degree,
    "diameter": diam,
    "average_shortest_path_length": avg_path,
    "is_directed": G.is_directed(),
}

if G.is_directed():
    result["reciprocity"] = nx.reciprocity(G)

pd.DataFrame([result])


## 5) Compute node-level measures (Link Analysis)

We compute:
- Degree centrality  
- Closeness centrality  
- Betweenness centrality  
- Eigenvector centrality  
- Local clustering coefficient  

Then we list the **Top 10** nodes for each measure.


In [ ]:
# Centrality measures
deg_cent = nx.degree_centrality(G)
close_cent = nx.closeness_centrality(G)
between_cent = nx.betweenness_centrality(G, normalized=True)
cluster_coeff = nx.clustering(G.to_undirected())  # defined for undirected; works on underlying undirected

# Eigenvector centrality can fail to converge for some graphs; handle safely
try:
    eig_cent = nx.eigenvector_centrality(G.to_undirected(), max_iter=2000)
except Exception as e:
    print("Eigenvector centrality failed:", e)
    eig_cent = {n: np.nan for n in G.nodes()}

metrics_df = pd.DataFrame({
    "node": list(G.nodes()),
    "degree_centrality": [deg_cent[n] for n in G.nodes()],
    "closeness_centrality": [close_cent[n] for n in G.nodes()],
    "betweenness_centrality": [between_cent[n] for n in G.nodes()],
    "eigenvector_centrality": [eig_cent[n] for n in G.nodes()],
    "local_clustering": [cluster_coeff[n] for n in G.nodes()],
})

metrics_df.head()


In [ ]:
def topk(df, col, k=10):
    return df.sort_values(col, ascending=False).head(k)[["node", col]]

for col in ["degree_centrality", "closeness_centrality", "betweenness_centrality", "eigenvector_centrality", "local_clustering"]:
    print("\nTop nodes by", col)
    display(topk(metrics_df, col, k=10))


## 6) Visualize the network

We will:
- Draw the network
- Highlight top 5 nodes by eigenvector centrality (influence)

> If your graph is large, consider drawing only a subgraph.


In [ ]:
# Option: sample a subgraph for plotting if too large
MAX_NODES_TO_PLOT = 300

H = G
if G.number_of_nodes() > MAX_NODES_TO_PLOT:
    # sample nodes by degree (keep hubs)
    nodes_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)
    keep_nodes = [n for n, _ in nodes_sorted[:MAX_NODES_TO_PLOT]]
    H = G.subgraph(keep_nodes).copy()
    print(f"Plotting subgraph with {H.number_of_nodes()} nodes and {H.number_of_edges()} edges")

# Identify top 5 influential nodes
top_influencers = topk(metrics_df, "eigenvector_centrality", k=5)["node"].tolist()

pos = nx.spring_layout(H, seed=RANDOM_SEED)

plt.figure(figsize=(10, 7))
nx.draw_networkx_edges(H, pos, alpha=0.25)
nx.draw_networkx_nodes(H, pos, node_size=60)
nx.draw_networkx_nodes(H, pos, nodelist=[n for n in top_influencers if n in H.nodes()], node_size=220)
nx.draw_networkx_labels(H, pos, labels={n: str(n) for n in top_influencers if n in H.nodes()}, font_size=10)
plt.axis("off")
plt.title("Network Visualization (Top Influencers Labeled)")
plt.show()

print("Top influencers (by eigenvector centrality):", top_influencers)


## 7) Community detection

We use **greedy modularity communities** (works well for undirected graphs).

For directed graphs, we apply it on the **undirected** version.


In [ ]:
G_und = G.to_undirected()

communities = list(community.greedy_modularity_communities(G_und))
print("Number of communities detected:", len(communities))

# Community sizes
sizes = [len(c) for c in communities]
pd.DataFrame({"community_id": range(len(communities)), "size": sizes}).sort_values("size", ascending=False).head(10)


In [ ]:
# Visualize communities (on a plot-friendly subgraph)
MAX_NODES_TO_PLOT = 300
H = G_und
if H.number_of_nodes() > MAX_NODES_TO_PLOT:
    nodes_sorted = sorted(H.degree(), key=lambda x: x[1], reverse=True)
    keep_nodes = [n for n, _ in nodes_sorted[:MAX_NODES_TO_PLOT]]
    H = H.subgraph(keep_nodes).copy()

# Map node -> community id
node_to_comm = {}
for i, comm in enumerate(communities):
    for n in comm:
        node_to_comm[n] = i

# Positions
pos = nx.spring_layout(H, seed=RANDOM_SEED)

# Build node sizes by degree (simple emphasis)
node_sizes = [50 + 10*H.degree(n) for n in H.nodes()]

plt.figure(figsize=(10, 7))
nx.draw_networkx_edges(H, pos, alpha=0.20)
nx.draw_networkx_nodes(
    H, pos,
    node_size=node_sizes
)
plt.axis("off")
plt.title("Community Visualization (structure only)")
plt.show()

print("Note: In Colab we avoid custom colors to keep plotting simple.")


## 8) Mini recommendation task: “Who should I connect to?”

A simple rule:
- Recommend nodes with **many mutual neighbors** (common neighbors).
- For directed graphs, we recommend based on undirected mutuals.

We generate Top-10 recommendations for a chosen node `u`.


In [ ]:
# Choose a node to recommend connections for:
u = list(G_und.nodes())[0]  # change this to any node ID you want

def recommend_by_common_neighbors(G_und, u, k=10):
    # Candidates: nodes not already connected to u
    neighbors = set(G_und.neighbors(u))
    candidates = set(G_und.nodes()) - neighbors - {u}

    scores = []
    for v in candidates:
        # common neighbors count
        common = len(list(nx.common_neighbors(G_und, u, v)))
        if common > 0:
            scores.append((v, common))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

recs = recommend_by_common_neighbors(G_und, u, k=10)
print("Recommendations for node:", u)
pd.DataFrame(recs, columns=["recommended_node", "mutual_neighbors"])


## 9) Robustness test (network resilience)

Remove the most influential node (by eigenvector centrality) and observe:
- change in number of connected components
- size of the largest connected component

This shows how much your network depends on a few hubs.


In [ ]:
# Pick the most influential node
most_influential = topk(metrics_df, "eigenvector_centrality", k=1)["node"].iloc[0]
print("Most influential node:", most_influential)

Before = G_und.copy()
After = G_und.copy()
if most_influential in After:
    After.remove_node(most_influential)

def largest_component_size(Gx):
    if Gx.number_of_nodes() == 0:
        return 0
    comps = list(nx.connected_components(Gx))
    return max(len(c) for c in comps)

before_components = nx.number_connected_components(Before) if Before.number_of_nodes() else 0
after_components = nx.number_connected_components(After) if After.number_of_nodes() else 0

summary = pd.DataFrame([{
    "before_components": before_components,
    "after_components": after_components,
    "before_largest_component": largest_component_size(Before),
    "after_largest_component": largest_component_size(After),
    "nodes_before": Before.number_of_nodes(),
    "nodes_after": After.number_of_nodes(),
    "edges_before": Before.number_of_edges(),
    "edges_after": After.number_of_edges()
}])

summary


## 10) Questions to answer (submit with your notebook)

1. **Top Influencer:** Who is the top node by eigenvector centrality? What might that mean in real life?  
2. **Bridge Node:** Which node has highest betweenness centrality? Why is it important?  
3. **Network Structure:** Is the network centralized or distributed? Use degree distribution / centrality evidence.  
4. **Communities:** How many communities did you detect? Describe the two largest communities.  
5. **Recommendation:** For one node `u`, explain why your Top-3 recommendations make sense.  
6. **Robustness:** After removing the most influential node, what changed? Interpret the result.

---

### Submission checklist
- ✅ Notebook runs end-to-end  
- ✅ No personal names/emails included (anonymized)  
- ✅ Figures included  
- ✅ Answers to Q1–Q6 included in a final markdown cell
